# Task 2 - Generative AI: Domain-Specific Fine-Tuning Pipeline

**Use case: Financial Compliance Clause Classifier** (see `src/dataset_generation.py`
docstring for the full problem statement). Requires:
- A Mistral AI API key (`MISTRAL_API_KEY` Colab secret) for the teacher model and LLM judge.
- A T4 GPU runtime (Runtime > Change runtime type > T4 GPU) for Task 2B.
- Optionally a Hugging Face token (`HF_TOKEN` Colab secret) to push the merged model.
- Optionally a Weights & Biases API key for training-loss logging.

In [ ]:
!pip install -q mistralai pydantic rouge_score bert_score
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets wandb

In [ ]:
import os
from google.colab import userdata
os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('HF_TOKEN not set -- merged model will be saved locally only')
import sys
sys.path.append('src')


## Task 2A - Dataset Generation and Diversity Report

In [ ]:
from dataset_generation import generate_dataset, report_diversity, split_dataset, to_chat_jsonl
import json

dataset = generate_dataset(n=150)
print(f'Generated {len(dataset)} examples')

diversity = report_diversity(dataset)
print(json.dumps(diversity, indent=2))

In [ ]:
train, val, test = split_dataset(dataset)
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

to_chat_jsonl(train, 'train.jsonl')
to_chat_jsonl(val, 'val.jsonl')
to_chat_jsonl(test, 'test.jsonl')

!head -2 train.jsonl

## Task 2B - QLoRA Fine-Tuning

**Hyperparameter justifications are documented as comments directly in
`src/finetune_qlora.py`** (LoRA r/alpha/target_modules, learning rate,
scheduler, epochs, batch size, gradient accumulation, max_seq_length).
This cell runs the full training loop and prints train/val loss per epoch.

In [ ]:
import wandb
wandb.login()  # or wandb.init(mode='disabled') to log to console only

from finetune_qlora import run_finetuning
trainer, merged_model = run_finetuning('train.jsonl', 'val.jsonl')

In [ ]:
# Train/val loss per epoch, pulled from the trainer's log history
for entry in trainer.state.log_history:
    if 'loss' in entry:
        print(f"step {entry.get('step')}: train_loss={entry['loss']:.4f}")
    if 'eval_loss' in entry:
        print(f"step {entry.get('step')}: eval_loss={entry['eval_loss']:.4f}  <-- must decrease across epochs")

## Task 2C - Evaluation and Baseline Comparison

In [ ]:
import json
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from evaluate import run_comparison, manual_review_and_hallucination_rate, llm_judge
from dataset_generation import CLASSIFIER_SYSTEM_PROMPT
from mistralai import Mistral

test_examples = [json.loads(l) for l in open('test.jsonl')]
references = [ex['messages'][2]['content'] for ex in test_examples]

# Base model: same architecture, system prompt only, NO fine-tuning
base_tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
base_model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct', device_map='auto')
base_pipe = pipeline('text-generation', model=base_model, tokenizer=base_tokenizer, max_new_tokens=200)

def generate(pipe, tokenizer, messages):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, do_sample=False)[0]['generated_text']
    return out[len(prompt):].strip()

base_predictions = [generate(base_pipe, base_tokenizer, ex['messages']) for ex in test_examples]

In [ ]:
ft_tokenizer = AutoTokenizer.from_pretrained('compliance-clause-classifier-lora-merged')
ft_model = AutoModelForCausalLM.from_pretrained('compliance-clause-classifier-lora-merged', device_map='auto')
ft_pipe = pipeline('text-generation', model=ft_model, tokenizer=ft_tokenizer, max_new_tokens=200)

finetuned_predictions = [generate(ft_pipe, ft_tokenizer, ex['messages']) for ex in test_examples]

In [ ]:
comparison = run_comparison(base_predictions, finetuned_predictions, references)

In [ ]:
# Additional metric: LLM-as-judge, structured JSON scoring
judge_client = Mistral(api_key=os.environ['MISTRAL_API_KEY'])
judge_scores = [llm_judge(judge_client, pred, ref) for pred, ref in zip(finetuned_predictions, references)]
valid_scores = [s for s in judge_scores if s is not None]
avg_overall = sum(s.overall for s in valid_scores) / len(valid_scores)
print(f'LLM-judge average overall score (fine-tuned): {avg_overall:.2f} / 5')
for s in valid_scores[:5]:
    print(s.model_dump())

In [ ]:
labels, hallucination_rate = manual_review_and_hallucination_rate(finetuned_predictions, min_reviewed=10)
print(f'Hallucination rate: {hallucination_rate:.1f}%')

### Qualitative Analysis (fill in after reviewing the outputs above)

**Where fine-tuning improved behaviour:** *[Write 1 paragraph citing specific
test-set examples where the fine-tuned model produced correctly-typed JSON
with a grounded flag_reason, versus the base model's looser/off-taxonomy
output.]*

**Remaining failure modes and next steps:** *[Write 1 paragraph on the
specific error patterns seen in the hallucination review above -- e.g.
confusion between adjacent clause types, risk_level miscalibration -- and
what additional data or training strategy (more examples of the confused
pair, harder negatives, a larger rank) would address them.]*